In [12]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
from collections import Counter
import re

In [13]:
docs = [
    "HELLO?? anyone here!!!",
    "I just bought 2 apples, 3 bananas, & a pear 🍐 from the store.",
    "uhhh... not sure... maybe tomorrow??",
    "What is love??? 🥲",
    "call me @ 9:30pm -- don’t b late!!!",
    "Th1s is s0me rand0m t3xt w1th numb3rs.",
    "Lolololol that was soooo funnyyyyy!!!! 😂😂😂",
    "USER_1234 left the chat at 17:46 on 03/12/2023.",
    "BUY NOW!!! limited time offer: https://scammy.link/abc123",
    "I'm... I'm not sure what to say anymore...",
    "RT @someone: OMG check this out!! #trending #viral",
    "okOkOk... brb gotta go eat 🍔",
    "Meeting rescheduled to Fri. 2pm, not Wed!",
    "⚠️ SYSTEM ERROR: Code 0x00ffdead – contact admin",
    "Did you see that movie??? 'Twilight of the Ducks'... 10/10",
    "the quick brown fox jumps ov3r the lazy d0g!",
    "Y'all ain't ready for this 🔥🔥🔥",
    "Just landed in 🇯🇵 Tokyo! #adventure #travel",
    "Weather's kinda meh today, like, cloudy but not raining?",
    "He said “No”, but I think he meant “Yes”… 🙃",
]


In [14]:
from transformers import AutoTokenizer, AutoModel

In [15]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

In [16]:
# Tokenize and encode the batch of text
inputs = tokenizer(docs, padding=True, truncation=True, return_tensors="pt")

# Forward pass through the model
outputs = model(**inputs)

# outputs.last_hidden_state is the embeddings for each token
print(outputs.last_hidden_state.shape)  # e.g., [batch_size, seq_len, hidden_dim]

torch.Size([20, 24, 768])


In [17]:
# Access the word index (token-to-ID mapping)
word_index = tokenizer.get_vocab()

# View the first few token-ID mappings
for token, index in list(word_index.items())[:10]:  # Show first 10 tokens
    print(f"Token: {token}, Index: {index}")

Token: haifa, Index: 21303
Token: ##¼, Index: 29664
Token: ##all, Index: 8095
Token: [unused241], Index: 246
Token: chest, Index: 3108
Token: sales, Index: 4341
Token: jaw, Index: 5730
Token: philanthropist, Index: 15246
Token: ##pping, Index: 14853
Token: evacuated, Index: 13377


In [18]:
# Tokenize and flatten the list of tokens
tokens = []
for text in docs:
    # Tokenize each sentence and convert to a list of tokens
    tokens += tokenizer.tokenize(text)

# Count the frequency of each token using Counter
word_counts = Counter(tokens)

# Display the 10 most common tokens
print(word_counts.most_common(10))

[('.', 23), ('!', 18), ('?', 11), ('[UNK]', 8), ("'", 7), (',', 6), (':', 6), ('/', 6), ('the', 5), ('##3', 5)]


In [19]:
from keras.datasets import imdb
from keras.utils import pad_sequences as keras_pad

(X_train, y_train), (X_test, y_test) = imdb.load_data()

# Pad sequences
X_train = keras_pad(X_train, padding='post', maxlen=50)
X_test = keras_pad(X_test, padding='post', maxlen=50)

X_train = torch.tensor(X_train).long()
y_train = torch.tensor(y_train).float()
X_test = torch.tensor(X_test).long()
y_test = torch.tensor(y_test).float()

# Convert to datasets
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [20]:
class SimpleRNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size):
        super(SimpleRNNModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = out[:, -1, :]  # Take last time step
        out = self.fc(out)
        return torch.sigmoid(out).squeeze()

In [21]:
vocab_size = max(max(X_train.flatten()).item(), max(X_test.flatten()).item()) + 1
model = SimpleRNNModel(vocab_size=vocab_size, embed_dim=32, hidden_size=32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 268.0015
Epoch 2, Loss: 242.7962
Epoch 3, Loss: 207.3898
Epoch 4, Loss: 179.4251
Epoch 5, Loss: 157.6492
Epoch 6, Loss: 142.1987
Epoch 7, Loss: 124.3665
Epoch 8, Loss: 110.1903
Epoch 9, Loss: 96.9778
Epoch 10, Loss: 87.9652
Epoch 11, Loss: 79.6745
Epoch 12, Loss: 70.4108
Epoch 13, Loss: 64.1987
Epoch 14, Loss: 56.0854
Epoch 15, Loss: 52.7289
Epoch 16, Loss: 46.6523
Epoch 17, Loss: 43.6630
Epoch 18, Loss: 58.1061
Epoch 19, Loss: 36.7292
Epoch 20, Loss: 33.1272
Epoch 21, Loss: 74.7633
Epoch 22, Loss: 74.6391
Epoch 23, Loss: 31.0633
Epoch 24, Loss: 24.5990
Epoch 25, Loss: 22.7164
Epoch 26, Loss: 21.1883
Epoch 27, Loss: 18.8301
Epoch 28, Loss: 16.8738
Epoch 29, Loss: 21.8867
Epoch 30, Loss: 22.4521
Epoch 31, Loss: 21.1047
Epoch 32, Loss: 12.0676
Epoch 33, Loss: 13.7980
Epoch 34, Loss: 10.1985
Epoch 35, Loss: 17.7161
Epoch 36, Loss: 14.4412
Epoch 37, Loss: 10.5027
Epoch 38, Loss: 10.2114
Epoch 39, Loss: 6.9698
Epoch 40, Loss: 12.1381
Epoch 41, Loss: 11.4922
Epoch 42, Loss: 9.

In [22]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        preds = model(xb)
        predicted = (preds > 0.5).float()
        correct += (predicted == yb).sum().item()
        total += yb.size(0)

print(f"Accuracy: {correct/total:.4f}")

Accuracy: 0.7552
